# Image-to-Image Translation

## Image-to-Image Translation

Image-to-image translation is a task in computer vision that involves learning a mapping from one visual domain to another. In our specific context, the goal is to convert microscopy images of cells that are out-of-focus into images that are in-focus. This type of task is essential in biological and medical imaging where image quality directly impacts downstream tasks such as cell counting, segmentation, and other forms of quantitative analysis. Restoring images from out-of-focus to in-focus is crucial for ensuring that the data used in these analyses is accurate and reliable.


## Pix2Pix
To solve the above mentioned issue, we use a Generative Adversarial Network (GAN) architecture called Pix2Pix.
Pix2Pix was introduced by Isola et al. in their 2017 paper "Image-to-Image Translation with Conditional Adversarial Networks." It specifically focuses on image-to-image translation tasks using paired images. The model consists of two primary components:

- Generator (G): This network generates images from input data.
- Discriminator (D): This network evaluates the generated images against real images, distinguishing between real and fake.

The training process is a competition between these two networks, where the generator improves to create more realistic images, while the discriminator enhances its ability to detect fake images.

The generator in this case is a U-Net, which is effective for tasks that involve spatial transformations of images. The discriminator, on the other hand, is a PatchGAN, which evaluates the realism of generated images at the level of patches rather than the entire image. This combination of U-Net and PatchGAN is highly effective for tasks where fine-grained details need to be preserved.

## About the Dataset

The dataset used in this project is a collection of microscopy images of Human U2OS cells. These images have been captured at varying focal planes, some of which are out-of-focus and some of which are in-focus. The images are provided in 16-bit TIFF format and are labeled as either 'in-focus' or 'out-of-focus.' Each image is 696x520 pixels in size.

## Outline
1. **Importing Necessary Libraries**
2. **Loading and Preprocessing the Dataset**
3. **Visualizing the Dataset**
4. **Building the Generator Network**
5. **Building the Discriminator Network**
6. **Defining Loss Functions and Optimizers**
7. **Training the GAN**
8. **Visualizing the Results**
9. **Saving the Model**
10. **Loading and Visualizing Predictions from the Saved Model**


## Step 1: Importing Necessary Libraries

In [ ]:
# Import necessary libraries
import numpy as np
from PIL import Image
import tensorflow as tf
from tensorflow.keras.layers import Conv2D, Conv2DTranspose, LeakyReLU, BatchNormalization, Dropout, ReLU, Concatenate, Input, ZeroPadding2D, Cropping2D
from tensorflow.keras.models import Model
import os
from glob import glob
import matplotlib.pyplot as plt


## Step 2: Loading and Preprocessing the Dataset

Loading and preprocessing the dataset is a critical step in preparing the data for training the neural network. The images in this dataset are in TIFF format, a high-quality image format widely used in scientific imaging due to its support for various pixel depths, including 16-bit images. However, to make these images suitable for training a deep learning model, we need to preprocess them. The `load_tiff_image` function is designed to handle this preprocessing. It first loads the image and converts it into a NumPy array. After loading, the function checks the shape of the image to ensure consistency. Since all images must have the same dimensions, the function transposes the image if necessary to match the expected shape of 696x520 pixels.

Once the shape is correct, the next step is normalization. Normalization is a crucial preprocessing step because it ensures that the input data is scaled consistently, which can significantly improve the performance and stability of the training process. In this case, the pixel values are normalized to the range [-1, 1], which is a common practice when working with GANs. This is because GANs use the `tanh` activation function in the output layer, which also produces outputs in the range [-1, 1]. By normalizing the inputs to this range, we help the model learn more effectively. After normalization, a channel dimension is added to the array, converting it into a 4D tensor (batch size, height, width, channels). This is necessary because convolutional layers in TensorFlow expect inputs to be in this format, even for grayscale images. Finally, the image data is cast to the `float32` data type, which is the standard for TensorFlow operations. Converting to `float32` ensures that the data is compatible with the model and optimizers during training.

The `load_and_preprocess_data` function is responsible for loading and preparing the entire dataset. This function starts by retrieving the file paths of all TIFF images from the specified directories. The images are separated into two categories: out-of-focus and in-focus images. This separation is critical because Pix2Pix, being a conditional GAN, requires paired examples of input and output images. For each out-of-focus image, the corresponding in-focus image is loaded, ensuring that the model has the correct training pairs. Each image is preprocessed using the `load_tiff_image` function, which standardizes the images by normalizing their pixel values and adding the necessary dimensions. After all the images have been processed, they are converted into NumPy arrays. Converting the images into arrays is essential because TensorFlow operates on NumPy arrays when training models. These arrays of images and their corresponding masks (in-focus images) are then returned, ready to be used for training the GAN. This function ensures that the dataset is loaded efficiently and preprocessed consistently, which is crucial for the performance of the model.



In [ ]:
# Function to load and preprocess TIFF images
def load_tiff_image(path):
    img = Image.open(path)
    img_array = np.array(img)

    # Ensure that the image has the correct shape (transpose if necessary)
    if img_array.shape != (696, 520):
        img_array = img_array.transpose(1, 0)  # Swap dimensions if necessary

    # Normalize the image to [-1, 1]
    img_array = img_array / np.max(img_array)  # Normalize to [0, 1]
    img_array = (img_array - 0.5) * 2  # Normalize to [-1, 1]

    # Add the channel dimension (1 for grayscale)
    img_array = np.expand_dims(img_array, axis=-1)

    # Convert to float32
    img_array = img_array.astype(np.float32)

    return img_array

In [ ]:
# Function to load and preprocess datasets from given directories
def load_and_preprocess_data(out_of_focus_dir, in_focus_dir):
    out_of_focus_files = sorted(glob(os.path.join(out_of_focus_dir, '**', '*.tif'), recursive=True))
    in_focus_files = sorted(glob(os.path.join(in_focus_dir, '**', '*.tif'), recursive=True))

    images = []
    masks = []

    for out_file, in_file in zip(out_of_focus_files, in_focus_files):
        image = load_tiff_image(out_file)
        mask = load_tiff_image(in_file)
        images.append(image)
        masks.append(mask)

    images = np.array(images)
    masks = np.array(masks)

    return images, masks

# Define the paths to your image directories
out_of_focus_path = 'BBBC006_v1_images_z_00'
in_focus_path = 'BBBC006_v1_images_z_16'

# Load and preprocess the datasets
images, masks = load_and_preprocess_data(out_of_focus_path, in_focus_path)

# Print the shapes to confirm
print(f"Image shape: {images.shape}")
print(f"Mask shape: {masks.shape}")


## Step 3: Visualizing the Dataset

Before training the model, it is important to visualize the data to ensure that it has been loaded and preprocessed correctly. Visualization allows us to inspect a sample of the images and verify that the input-output pairs are correctly aligned. The `visualize_data` function is designed to display both the out-of-focus and in-focus images side by side, providing a clear comparison between the input and target images that the model will be trained on. By displaying the images in grayscale, we ensure that the full range of pixel intensities is visible, allowing us to verify that the preprocessing steps have been applied correctly and that the images are in the expected format.


In [ ]:
def visualize_data(images, masks, num_samples=5):
    if len(images) == 0 or len(masks) == 0:
        print("No images to display. Please check the image paths and loading process.")
        return

    plt.figure(figsize=(25, 10))
    for i in range(min(num_samples, len(images))):  # Use min to avoid out-of-bounds indexing
        image, mask = images[i], masks[i]

        plt.subplot(2, num_samples, i + 1)
        plt.imshow(image, cmap='gray')  # Display the out-of-focus image in grayscale
        plt.title("Out-of-Focus Image " + str(i + 1))
        plt.axis('off')

        plt.subplot(2, num_samples, i + 1 + num_samples)
        plt.imshow(mask, cmap='gray')  # Display the in-focus image in grayscale
        plt.title("In-Focus Image " + str(i + 1))
        plt.axis('off')

    plt.show()  # Show the plot with the images

# Visualize the first few samples from the dataset
visualize_data(images, masks)


## Step 4: Building the U-Net Generator

The generator is the core component of the Pix2Pix model that performs the image-to-image translation. In this implementation, we use a U-Net architecture for the generator, which is particularly effective for tasks that require fine-grained transformations, such as converting out-of-focus images into in-focus images. The U-Net architecture consists of an encoder-decoder structure with skip connections between corresponding layers in the encoder and decoder. These skip connections help the model preserve spatial information, which is crucial for generating high-quality images that retain the details of the original input.

The `downscale` function defines the downsampling blocks of the U-Net encoder. Each downsampling block consists of a convolutional layer, followed by batch normalization (optional) and a LeakyReLU activation function. The convolutional layer reduces the spatial dimensions of the input while increasing the number of feature maps, allowing the model to capture higher-level features as the image is passed through successive layers. The use of LeakyReLU instead of a standard ReLU activation helps prevent the problem of "dying neurons" by allowing a small gradient to pass through even for negative input values.

The `upscale` function defines the upsampling blocks of the U-Net decoder. Each upsampling block consists of a transposed convolutional layer, batch normalization, and a ReLU activation function. The transposed convolutional layer increases the spatial dimensions of the input, effectively "undoing" the downsampling performed by the encoder. Dropout is optionally applied in some of the upsampling blocks to prevent overfitting. The skip connections are implemented by concatenating the feature maps from the encoder with the corresponding feature maps in the decoder. This helps the model recover spatial details that might have been lost during the downsampling process.

The `create_generator` function assembles the full U-Net architecture. The input is passed through the downsampling blocks, and the resulting feature maps are stored for later use in the skip connections. The upsampling blocks then reconstruct the image, and the final output layer uses a transposed convolutional layer with a `tanh` activation function to produce the generated image. The `tanh` activation ensures that the output pixel values are in the range [-1, 1], matching the normalized range of the input images. By using this architecture, the generator is able to produce high-quality in-focus images from out-of-focus inputs, making it a powerful tool for image-to-image translation tasks in bioimaging.


In [ ]:
# Downscaling block for the U-Net generator
def downscale(num_filters, apply_batchnorm=True):
    block = tf.keras.Sequential()
    block.add(Conv2D(num_filters, kernel_size=4, strides=2, padding='same', kernel_initializer='he_normal', use_bias=False))
    if apply_batchnorm:
        block.add(BatchNormalization())
    block.add(LeakyReLU(alpha=0.2))
    return block

# Upscaling block for the U-Net generator
def upscale(num_filters, apply_dropout=False):
    block = tf.keras.Sequential()
    block.add(Conv2DTranspose(num_filters, kernel_size=4, strides=2, padding='same', kernel_initializer='he_normal', use_bias=False))
    block.add(BatchNormalization())
    if apply_dropout:
        block.add(Dropout(0.5))
    block.add(ReLU())
    return block

# Create the U-Net generator
def create_generator():
    inputs = Input(shape=[696, 520, 1])

    # Encoder (downsampling path)
    down_stack = [
        downscale(64, apply_batchnorm=False),
        downscale(128),
        downscale(256),
        downscale(512),
        downscale(512),
        downscale(512),
        downscale(512),
        downscale(512),
    ]

    # Decoder (upsampling path)
    up_stack = [
        upscale(512, apply_dropout=True),
        upscale(512, apply_dropout=True),
        upscale(512, apply_dropout=True),
        upscale(512),
        upscale(256),
        upscale(128),
        upscale(64),
    ]

    initializer = tf.random_normal_initializer(0., 0.02)
    last = Conv2DTranspose(1, kernel_size=4, strides=2, padding='same', kernel_initializer=initializer, activation='tanh')

    x = inputs

    # Downsampling through the model
    skips = []
    for down in down_stack:
        x = down(x)
        skips.append(x)

    skips = reversed(skips[:-1])

    # Upsampling and establishing the skip connections
    for up, skip in zip(up_stack, skips):
        x = up(x)

        # Apply padding if necessary to ensure the dimensions match
        if x.shape[1] != skip.shape[1] or x.shape[2] != skip.shape[2]:
            if x.shape[1] > skip.shape[1]:
                x = Cropping2D(((0, x.shape[1] - skip.shape[1]), (0, 0)))(x)
            elif x.shape[1] < skip.shape[1]:
                skip = Cropping2D(((0, skip.shape[1] - x.shape[1]), (0, 0)))(skip)

            if x.shape[2] > skip.shape[2]:
                x = Cropping2D(((0, 0), (0, x.shape[2] - skip.shape[2])))(x)
            elif x.shape[2] < skip.shape[2]:
                skip = Cropping2D(((0, 0), (0, skip.shape[2] - x.shape[2])))(skip)

        x = Concatenate()([x, skip])

    x = last(x)

    return Model(inputs=inputs, outputs=x)

# Create the generator and display its architecture
generator = create_generator()
generator.summary()


## Step 5: Building the PatchGAN Discriminator

The discriminator in the Pix2Pix model is designed to distinguish between real in-focus images and fake (generated) in-focus images. Instead of evaluating the entire image at once, the discriminator uses a PatchGAN architecture, which operates on smaller patches of the image. This allows the discriminator to focus on local details and make decisions based on high-frequency components, which are crucial for tasks like assessing image sharpness and focus quality. The PatchGAN discriminator is especially effective for tasks where fine details matter, as it evaluates the realism of the generated images at a more granular level.

The `create_discriminator` function defines the architecture of this PatchGAN discriminator. The input to the discriminator consists of two images: the out-of-focus image (input) and either the real in-focus image (target) or the generated in-focus image (output from the generator). These two images are concatenated along the channel dimension to create a combined input that is fed into the discriminator. The architecture of the discriminator is similar to that of the encoder in the U-Net generator, with several downsampling blocks that reduce the spatial dimensions of the image while increasing the number of feature maps. These blocks consist of convolutional layers, batch normalization, and LeakyReLU activations. The final layer outputs a single-channel feature map, where each value represents the likelihood of the corresponding patch being real or generated. This patch-based approach helps the model focus on local image details, leading to better overall performance in tasks that require attention to fine-grained features.


In [ ]:
# Function to create the PatchGAN discriminator
def create_discriminator():
    initializer = tf.random_normal_initializer(0., 0.02)

    inp = Input(shape=[696, 520, 1], name='input_image')  # Input for out-of-focus images
    tar = Input(shape=[696, 520, 1], name='target_image')  # Input for corresponding in-focus images

    x = Concatenate()([inp, tar])

    down1 = downscale(64, apply_batchnorm=False)(x)
    down2 = downscale(128)(down1)
    down3 = downscale(256)(down2)

    zero_pad1 = tf.keras.layers.ZeroPadding2D()(down3)
    conv = Conv2D(512, kernel_size=4, strides=1, kernel_initializer=initializer, use_bias=False)(zero_pad1)
    batchnorm1 = BatchNormalization()(conv)
    leaky_relu = LeakyReLU(alpha=0.2)(batchnorm1)
    zero_pad2 = tf.keras.layers.ZeroPadding2D()(leaky_relu)

    last = Conv2D(1, kernel_size=4, strides=1, kernel_initializer=initializer)(zero_pad2)

    return Model(inputs=[inp, tar], outputs=last)

# Create the discriminator and display its architecture
discriminator = create_discriminator()
discriminator.summary()

## Step 6: Defining Loss Functions and Optimizers

In a GAN, both the generator and the discriminator have their own loss functions, which are used to guide the training process. The generator's loss is a combination of two components: the adversarial loss and the L1 loss. The adversarial loss encourages the generator to produce images that are indistinguishable from real images, by measuring how well the generated images fool the discriminator. This loss is calculated using binary cross-entropy, where the generator tries to maximize the probability that the discriminator classifies its output as real. The L1 loss, on the other hand, measures the pixel-wise difference between the generated image and the ground truth in-focus image. This loss encourages the generator to produce images that are not only realistic but also close to the actual target images. The combination of adversarial and L1 losses helps the generator balance realism with accuracy, ensuring that the output images are both visually convincing and closely aligned with the ground truth.

The discriminator's loss, also calculated using binary cross-entropy, measures how well the discriminator distinguishes between real and generated images. The discriminator tries to minimize the probability of classifying real images as fake and generated images as real. The total discriminator loss is the sum of two components: the loss for real images and the loss for generated images. By optimizing this loss, the discriminator becomes better at distinguishing between real and generated images, making the generator's task progressively more challenging.

The `generator_loss` and `discriminator_loss` functions implement these loss calculations. The Adam optimizer is used for both the generator and the discriminator, with a learning rate of 2e-4 and a beta1 value of 0.5. These hyperparameters are commonly used in GAN training to ensure stable convergence. The Adam optimizer is particularly well-suited for training deep networks like GANs because it adapts the learning rate for each parameter, helping the model converge more efficiently.


In [ ]:
# Define the loss functions and optimizers for training the GAN
LAMBDA = 100  # Weight for the L1 loss component

# Generator loss combines adversarial loss and L1 loss
def generator_loss(disc_generated_output, gen_output, target):
    gan_loss = tf.keras.losses.BinaryCrossentropy(from_logits=True)(tf.ones_like(disc_generated_output), disc_generated_output)

    # Ensure both target and gen_output are float32
    target = tf.cast(target, tf.float32)
    gen_output = tf.cast(gen_output, tf.float32)

    l1_loss = tf.reduce_mean(tf.abs(target - gen_output))
    total_gen_loss = gan_loss + (LAMBDA * l1_loss)
    return total_gen_loss

# Discriminator loss considers the real vs generated images
def discriminator_loss(disc_real_output, disc_generated_output):
    real_loss = tf.keras.losses.BinaryCrossentropy(from_logits=True)(tf.ones_like(disc_real_output), disc_real_output)
    generated_loss = tf.keras.losses.BinaryCrossentropy(from_logits=True)(tf.zeros_like(disc_generated_output), disc_generated_output)
    total_disc_loss = real_loss + generated_loss
    return total_disc_loss

# Define optimizers for the generator and discriminator
generator_optimizer = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)
discriminator_optimizer = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)


## Step 7: Training the GAN

The training process for a GAN involves alternating between updating the generator and the discriminator. In each iteration, the generator creates a new image based on the input (out-of-focus image), and both the real (in-focus) and generated images are passed through the discriminator. The goal of the generator is to produce images that are as close to the real images as possible, while the discriminator tries to correctly classify real and generated images. The `train_step` function implements a single training step, which includes generating an image, calculating the generator and discriminator losses, and updating their weights using the gradients of these losses.

During training, the generator and discriminator are updated in an alternating fashion. First, the generator's gradients are computed and applied using the Adam optimizer. Then, the discriminator's gradients are computed and applied. The model is trained for a specified number of epochs, with each epoch representing a complete pass through the entire dataset. The `fit` function defines this training loop, iterating over the dataset for the given number of epochs and calling the `train_step` function for each batch of images. The generator and discriminator losses are printed after each epoch to monitor the progress of the training process. By alternating the updates to the generator and discriminator, the model gradually improves its ability to produce realistic in-focus images from out-of-focus inputs.


In [ ]:
# Define the number of epochs for training
EPOCHS = 15

# Single training step function for one batch
def train_step(input_image, target, epoch):
    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        gen_output = generator(input_image, training=True)  # Generate images

        disc_real_output = discriminator([input_image, target], training=True)  # Real images
        disc_generated_output = discriminator([input_image, gen_output], training=True)  # Generated images

        # Calculate the generator and discriminator losses
        gen_loss = generator_loss(disc_generated_output, gen_output, target)
        disc_loss = discriminator_loss(disc_real_output, disc_generated_output)

    # Compute gradients for both the generator and discriminator
    generator_gradients = gen_tape.gradient(gen_loss, generator.trainable_variables)
    discriminator_gradients = disc_tape.gradient(disc_loss, discriminator.trainable_variables)

    # Apply gradients to the optimizers
    generator_optimizer.apply_gradients(zip(generator_gradients, generator.trainable_variables))
    discriminator_optimizer.apply_gradients(zip(discriminator_gradients, discriminator.trainable_variables))

    print(f'Epoch: {epoch}, Gen Loss: {gen_loss.numpy()}, Disc Loss: {disc_loss.numpy()}')

# Training loop over the entire dataset
def fit(dataset, epochs):
    """
    Fit the model on the dataset for a given number of epochs.
    """
    for epoch in range(epochs):
        print(f'Starting epoch {epoch+1}/{epochs}')
        for input_image, target in dataset:
            train_step(input_image, target, epoch)
        print(f'Completed epoch {epoch+1}/{epochs}')

# Prepare the dataset for training
BUFFER_SIZE = 400
BATCH_SIZE = 1


images, masks = load_and_preprocess_data(out_of_focus_path, in_focus_path)

dataset = tf.data.Dataset.from_tensor_slices((images, masks)).shuffle(BUFFER_SIZE).batch(BATCH_SIZE)

# Train the model for the specified number of epochs
fit(dataset, EPOCHS)

## Step 8: Visualizing the Results

After training the GAN, it is important to evaluate its performance by visualizing the generated images. The `show_predictions` function is designed to display a few randomly selected samples from the dataset, along with the corresponding generated images produced by the trained generator. This function helps in assessing how well the model has learned the task of transforming out-of-focus images into in-focus images. The function displays three images for each sample: the original out-of-focus image, the ground truth in-focus image, and the predicted in-focus image generated by the model. By visually comparing these images, we can get a qualitative sense of how well the model is performing.


In [ ]:
# Function to visualize model predictions after training
def show_predictions(num_samples):
    for i in range(num_samples):
        idx = np.random.randint(images.shape[0])
        image, mask = images[idx], masks[idx]
        predicted = generator.predict(tf.expand_dims(image, axis=0))[0]

        plt.figure(figsize=(10, 8))

        plt.subplot(1, 3, 1)
        plt.imshow(image, cmap='gray')
        plt.title("Out-of-Focus Image " + str(i + 1))
        plt.axis('off')

        plt.subplot(1, 3, 2)
        plt.imshow(mask, cmap='gray')
        plt.title("In-Focus Image " + str(i + 1))
        plt.axis('off')

        plt.subplot(1, 3, 3)
        plt.imshow(predicted, cmap='gray')
        plt.title("Predicted Image " + str(i + 1))
        plt.axis('off')

        plt.show()

# Visualize the model predictions on a few samples
show_predictions(5)


## Step 9: Saving the Model

Once the model has been trained, it is important to save the generator so that it can be reused later without having to retrain it from scratch. The `generator.save()` function is used to save the trained generator model to a file. This file includes the model's architecture and the learned weights, allowing us to reload and use the model for inference or further training at a later time. By saving the model, we ensure that the results of our work are preserved and can be reproduced or extended in the future. This step is particularly useful for deploying the model in a real-world application, where it can be loaded and used to process new out-of-focus images without needing to retrain the model each time.


In [ ]:
generator.save("model.h5")

## Step 10: Loading and Visualizing Predictions from the Saved Model

After saving the trained generator model, it is essential to validate its performance by loading the saved model and visualizing its predictions. This step involves using the `tf.keras.models.load_model()` function to reload the generator from the saved file, ensuring that the model's architecture and learned weights are restored for inference. By reusing the saved model, we can generate predictions on new or existing data without the need for additional training. The `visualize_saved_model_predictions()` function is designed to facilitate this process by randomly selecting a few images from the dataset, feeding the out-of-focus images into the generator, and displaying the predicted in-focus images alongside the original out-of-focus and ground truth images. This visualization helps assess the model's performance after being reloaded, allowing for a direct comparison between the model's predictions and the true in-focus images. This process confirms that the saved model performs as expected and that the generated results remain consistent with those seen during training.


In [ ]:
# Load the saved model
generator = tf.keras.models.load_model('model.h5')

# Function to visualize a few predictions using the saved model
def visualize_saved_model_predictions(images, masks, num_samples=5):
    for i in range(num_samples):
        idx = np.random.randint(images.shape[0])
        image, mask = images[idx], masks[idx]
        predicted = generator.predict(tf.expand_dims(image, axis=0))[0]

        plt.figure(figsize=(10, 8))

        plt.subplot(1, 3, 1)
        plt.imshow(image, cmap='gray')
        plt.title(f"Out-of-Focus Image {i+1}")
        plt.axis('off')

        plt.subplot(1, 3, 2)
        plt.imshow(mask, cmap='gray')
        plt.title(f"In-Focus Image {i+1}")
        plt.axis('off')

        plt.subplot(1, 3, 3)
        plt.imshow(predicted, cmap='gray')
        plt.title(f"Predicted Image {i+1}")
        plt.axis('off')

        plt.show()

# Visualize predictions using the saved model
visualize_saved_model_predictions(images, masks, num_samples=5)
